In [1]:
import os
import random
import time
import pickle
import pandas as pd
import geopandas as gpd
import shapely.geometry

from tqdm import tqdm
from streetview import StreetViewDownloader, ImageService
from IPython.display import Image, display
from typing import Dict, Tuple, List, Union, Any

from streetview import *
from socioeconomics import *
from building import *
from station import *
from prompt import *
from config import *

### 🌍 FILES INPUT

This notebook processes earthquake-related data and requires the following files based on `config.py`. The notebook will generate geospatial image samples and analysis-ready data for earthquake damage assessment. 

#### Earthquake Data

- **Community-Reported Intensity**  
  `2019_ridgecrest_DYFI.csv`  
  *"Did You Feel It?"* (DYFI) dataset containing self-reported earthquake intensity levels.

- **Seismic Station Measurements**  
  `2019-ridgecrest/stationlist.json`  
  JSON file with seismic station metadata and ground motion values.


#### Geographic Files

- **ZIP Code Boundaries**  
  `nhgis_shape/US_zcta_2019.shp`

- **Census Block Group Boundaries**  
  `nhgis_shape/US_blck_grp_2019_84.shp`

- **Socioeconomic Attributes**  
  `nhgis_shape/cbg_information.csv`  
  Contains demographic and economic indicators at the block group level.


In [3]:
df = pd.read_csv(DYFI_DATA)
df = df[df['Country'] == 'United States of America']
df = df.sort_values(by='Responses', ascending=False)
df["Zip Code"] = df["Zip Code"].apply(lambda x: str(int(x)).zfill(5))

# df1 - Generate Samples
df1 = df.iloc[:NUM_SAMPLES]

# df2 - Generate Dictionary for RAG
df2 = df.iloc[NUM_SAMPLES:][df['Responses']>=20]
df2 = df2.rename(columns={"Latitude": "latitude", "Longitude": "longitude"})
print(len(df1), len(df2))
df1

100 313


/var/folders/l0/f3brd6kd23d1j64rj2yz5r100000gp/T/ipykernel_50338/1240944628.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2 = df.iloc[NUM_SAMPLES:][df['Responses']>=20]


,City,State/Region,Country,Zip Code,MMI,Responses,Distance,Latitude,Longitude
1027,Ridgecrest,CA,United States of America,93555,VII,314,3 km,35.691731,-117.449167
220,Las Vegas,NV,United States of America,89101,IV,204,213 km,36.172608,-115.122034
630,San Diego,CA,United States of America,92101,III,179,316 km,32.723359,-117.169286
286,Los Angeles,CA,United States of America,90012,IV,109,185 km,34.065853,-118.238610
1082,Fresno,CA,United States of America,93721,III,108,205 km,36.732871,-119.783726
...,...,...,...,...,...,...,...,...,...
599,Encinitas,CA,United States of America,92024,III,49,279 km,33.058794,-117.255585
838,Mission Viejo,CA,United States of America,92692,IV,49,219 km,33.609913,-117.643124
966,Bakersfield,CA,United States of America,93311,IV,49,153 km,35.189205,-119.181329
309,Los Angeles,CA,United States of America,90036,IV,48,189 km,34.070369,-118.350102


In [4]:
eq_data = EARTHQUAKE_PARAMETERS

zcta = gpd.read_file(ZCTA_SHAPEFILE)
zcta = zcta.to_crs(epsg=4326)

socioeconomic_df = pd.read_csv(SOCIOECONOMIC_DATA)
cbg_gdf = gpd.read_file(CBG_SHAPEFILE)
cbg_gdf = cbg_gdf.to_crs(epsg=4326)

stations_df = load_station_data(STATION_DATA)
stations_df = stations_df.sort_values(by='Nresp', ascending=False)
stations_df.head(20)

,latitude,longitude,Intensity,Vs30,Nresp
3509,36.1708,-115.1376,3.8,306.57,322
416,36.7387,-119.7831,3.5,260.07,189
2937,32.7131,-117.1654,3.2,760.00,150
737,35.3660,-119.0198,4.1,268.49,142
2204,33.6847,-117.8253,4.0,760.00,104
3616,36.0331,-114.9855,3.6,760.00,103
414,34.4151,-119.7038,3.4,397.03,96
1747,34.7012,-118.1410,4.7,293.71,95
471,36.8218,-119.6964,3.5,259.54,93
1937,33.6834,-117.9979,4.1,760.00,89


### 🌍 Spatial Image Download

The following module generates random sampling points within geographic boundaries based on zipcode and downloads corresponding Google Street View images.


#### ⚙️ Functions

##### `generate_points_with_images`

Generates random points within a polygon that have valid Street View imagery.

- Handles coordinate selection, metadata recording, and image storage
- Supports resume capability and real-time progress saving


##### `download_location_images`

Downloads Google Street View images for specific coordinates.

- Manages file naming conventions
- Includes robust error handling for failed downloads


##### `generate_samples_with_images`

Processes multiple ZIP codes to collect geographic sample points with valid Street View images.

- Skips ZIP codes that already have sufficient samples
- Validates and cleans geographic input data


In [4]:
def generate_points_with_images(
    polygon, 
    downloader, 
    zip_code, 
    city, 
    state, 
    country, 
    points_needed, 
    output_csv_path,
    max_attempts=MAX_ATTEMPTS
):
    """
    Generate random points within a polygon that have valid Street View images.
    Save results to CSV in real-time as each valid point is found.
    """
    
    valid_points = []
    minx, miny, maxx, maxy = polygon.bounds
    attempts = 0
    valid_count = 0
    
    # Create directory for images
    os.makedirs('ridgecrest-images', exist_ok=True)

    pbar = tqdm(total=points_needed, desc=f"Finding points with images in {zip_code}")
    
    # Check if output CSV already exists and load existing data
    if os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path)
        # Filter for points from current zip code to know how many we already have
        zip_points = existing_df[existing_df['Zip Code'] == zip_code]
        valid_count = len(zip_points)
        valid_points = zip_points.to_dict('records')
        
        if valid_count >= points_needed:
            print(f"Already have {valid_count} points for ZIP {zip_code}. Skipping.")
            pbar.update(points_needed)
            pbar.close()
            return valid_points
        else:
            pbar.update(valid_count)
            points_needed = points_needed - valid_count
    
    while len(valid_points) < points_needed + valid_count and attempts < max_attempts:
        # Generate a random point within the bounding box
        pt = shapely.geometry.Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        
        # Check if point is within the polygon
        if polygon.contains(pt):
            lat, lng = pt.y, pt.x 
            temp_id = f"{zip_code}_temp_{attempts}"
            success, result = download_location_images(
                downloader,
                lat,
                lng,
                location_id=temp_id,
                services=[ImageService.GOOGLE],
                size='640x640'
            )
            
            if success:
                valid_count += 1
                sequential_id = f"{zip_code}_{valid_count}"

                if os.path.exists(result):
                    new_path = f"ridgecrest-images/{sequential_id}_google_streetview_{lat}_{lng}.jpg"
                    
                    try:
                        with open(result, 'rb') as src_file:
                            content = src_file.read()
                            with open(new_path, 'wb') as dest_file:
                                dest_file.write(content)
                        os.remove(result)
                        result = new_path
                    except Exception as e:
                        print(f"Error renaming file: {str(e)}")
                
                # Add the point with the sequential ID
                point_data = {
                    "location_id": sequential_id,
                    "City": city,
                    "State/Region": state,
                    "Country": country,
                    "Zip Code": zip_code,
                    "Latitude": lat,
                    "Longitude": lng,
                    "image_path": result
                }
                valid_points.append(point_data)
                
                # Save the updated dataframe in real-time
                new_df = pd.DataFrame([point_data])
                if os.path.exists(output_csv_path):
                    new_df.to_csv(output_csv_path, mode='a', header=False, index=False)
                else:
                    new_df.to_csv(output_csv_path, mode='w', header=True, index=False)
                
                pbar.update(1)
                
        attempts += 1
        
        # If too many attempts, break and report
        if attempts >= max_attempts and len(valid_points) < points_needed + valid_count:
            print(f"\nWarning: Could only find {len(valid_points)} valid points with images " 
                  f"for ZIP {zip_code} after {max_attempts} attempts.")
            break
    
    pbar.close()
    return valid_points


def download_location_images(
    downloader,
    latitude: float,
    longitude: float,
    location_id: str = None,
    services = None,
    size: str = '640x640'
):
    """
    Download images for a specific location using the specified services.
    Returns success status and image path or error message.
    """
    
    # Define the expected image path with location_id
    image_path = f"ridgecrest-images/{location_id}_google_streetview_{latitude}_{longitude}.jpg"
    
    # Download the image
    results = downloader.download_images(
        latitude=latitude,
        longitude=longitude,
        services=services,
        size=size
    )
    
    # Check if Google download was successful
    if 'google' in results and results['google'][0]:
        success, original_path = results['google']
    
        if location_id and location_id not in original_path:
            try:
                # Rename the file to include the location_id
                with open(original_path, 'rb') as src_file:
                    content = src_file.read()
                    with open(image_path, 'wb') as dest_file:
                        dest_file.write(content)
                if original_path != image_path:
                    os.remove(original_path)
                return True, image_path
            except Exception as e:
                return False, f"Error renaming file: {str(e)}"
        else:
            return True, original_path
    else:
        error_msg = results.get('google', (False, "Unknown error"))[1]
        return False, error_msg


def generate_samples_with_images(df1, zcta, downloader, points_per_zip, output_csv_path):
    """
    Generate samples with valid images for each ZIP code in df1.
    Save results to CSV in real-time.
    """
    # Check if output CSV already exists
    if os.path.exists(output_csv_path):
        all_samples = pd.read_csv(output_csv_path).to_dict('records')
        print(f"Found existing CSV with {len(all_samples)} samples. Will resume from there.")
    else:
        all_samples = []
    
    for _, row in df1.iterrows():
        zip_code = row["Zip Code"]
        
        # Check if we already have enough samples for this ZIP
        if os.path.exists(output_csv_path):
            existing_df = pd.read_csv(output_csv_path)
            zip_points = existing_df[existing_df['Zip Code'] == zip_code]
            if len(zip_points) >= points_per_zip:
                print(f"Already have {len(zip_points)} points for ZIP {zip_code}. Skipping.")
                continue
        
        poly_match = zcta[zcta["ZCTA5CE10"] == zip_code]

        if poly_match.empty:
            print(f"ZIP {zip_code} not found in shapefile. Skipping.")
            continue

        polygon = poly_match.geometry.values[0]
        
        # Generate points with images for this polygon
        valid_points = generate_points_with_images(
            polygon=polygon,
            downloader=downloader,
            zip_code=zip_code,
            city=row["City"],
            state=row["State/Region"],
            country=row["Country"],
            points_needed=points_per_zip,
            output_csv_path=output_csv_path
        )
        
        print(f"Collected {len(valid_points)} points with images for {row['City']} ZIP {zip_code}")
    
    # Load the final complete DataFrame
    if os.path.exists(output_csv_path):
        df_samples = pd.read_csv(output_csv_path)
        
        # Reorder columns to make location_id the first column if not already
        if 'location_id' in df_samples.columns:
            cols = df_samples.columns.tolist()
            cols.remove('location_id')
            new_cols = ['location_id'] + cols
            df_samples = df_samples[new_cols]
            df_samples.to_csv(output_csv_path, index=False)
        
        print(f"Successfully generated {len(df_samples)} total samples with images")
        return df_samples
    else:
        print("No samples were generated.")
        return pd.DataFrame()

In [ ]:
downloader = StreetViewDownloader()
samples_df1 = generate_samples_with_images(df1, zcta, downloader, points_per_zip=POINTS_PER_ZIP, output_csv_path=OUTPUT_IMAGES_CSV)

### 🌍 Parameter Augmentation

This module adds geospatial and demographic context to earthquake-related datasets.

#### ⚙️ Functions

##### `add_parameter_columns()`

Adds earthquake-relevant parameters to a DataFrame using batched processing with checkpointing.

- Enriches each row with:
  - Earthquake metadata (location, magnitude)
  - Distance to epicenter
  - Closest VS30 station data
  - Census-based sociodemographics
  - Building infrastructure info

In [ ]:
def add_parameter_columns(df, eq_data, max_retries=3, sleep_time=3, batch_size=10, checkpoint_file=CHECKPOINT_FILE):
    """
    Adds earthquake parameter columns to dataframe with batch processing and checkpointing.
    """
    # Load checkpoint if exists
    start_idx = 0
    result_df = df.copy().reset_index(drop=True)
    
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'rb') as f:
                checkpoint = pickle.load(f)
                result_df = checkpoint['df']
                start_idx = checkpoint['last_processed_idx'] + 1
                print(f"Resuming from index {start_idx}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
    
    # Initialize parameter columns
    parameter_columns = [
        'eq_place', 'eq_lat', 'eq_lng', 'eq_magnitude', 'distance', 'vs30',
        'population_density', 'urban_population_pct', 'median_household_income', 
        'education', 'over_65_rate', 'building'
    ]
    
    for col in parameter_columns:
        if col not in result_df.columns:
            result_df[col] = None
    
    # Pre-populate earthquake data for all rows
    result_df['eq_place'] = eq_data['place']
    result_df['eq_lat'] = round(eq_data['lat'], 3)
    result_df['eq_lng'] = round(eq_data['lng'], 3)
    result_df['eq_magnitude'] = round(eq_data['magnitude'], 1)
    
    # Process in batches
    total_rows = len(df)
    for batch_start in range(start_idx, total_rows, batch_size):
        batch_end = min(batch_start + batch_size, total_rows)
        print(f"\nProcessing batch: {batch_start} to {batch_end-1} of {total_rows}")
        
        for index in range(batch_start, batch_end):
            row = df.iloc[index]
            
            # Extract location data
            latitude = row['latitude']
            longitude = row['longitude']
            
            print(f"\nProcessing {index}/{total_rows}: {latitude}, {longitude}")
            
            # Get distance data
            distance = haversine_distance(latitude, longitude, eq_data['lat'], eq_data['lng'])
            result_df.loc[index, 'distance'] = round(distance, 2)

            # Get station data
            closest_station = find_closest_station(latitude, longitude, stations_df)
            result_df.loc[index, 'vs30'] = closest_station.get('vs30')

            # Get census data
            try:
                census = get_sociodemographics(latitude, longitude, socioeconomic_df, cbg_gdf)
                if census and census.get("status") == "success":
                    result_df.loc[index, 'population_density'] = round(census['Population_Density'], 2)
                    result_df.loc[index, 'urban_population_pct'] = round(census['Urbanized_Areas_Population_R'], 2)
                    result_df.loc[index, 'median_household_income'] = round(census['Median_income'], 0)
                    result_df.loc[index, 'education'] = round(census['Education_Degree_R'], 2)
                    result_df.loc[index, 'over_65_rate'] = round(census['Over_65_R'], 2)
                else:
                    print(f"Census data unavailable for {latitude}, {longitude}")
            except Exception as e:
                print(f"Error getting census data for {latitude}, {longitude}: {str(e)}")
                
            # Get building data
            try:
                building_result = get_building_info(latitude, longitude, radius=100, 
                                                  max_retries=max_retries, sleep_time=sleep_time)
                if building_result is not None:
                    building = describe_buildings(building_result, radius=100)
                    result_df.loc[index, 'building'] = str(building)
                else:
                    result_df.loc[index, 'building'] = "Building information is not available."
            except Exception as e:
                print(f"Error getting building info for {latitude}, {longitude}: {str(e)}")
                result_df.loc[index, 'building'] = "Building information is not available."
            
            # Save checkpoint after each row
            checkpoint = {
                'df': result_df,
                'last_processed_idx': index
            }
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)
        
        # Add a pause between batches
        if batch_end < total_rows:
            pause_time = 30
            print(f"Completed batch. Pausing for {pause_time} seconds before next batch...")
            time.sleep(pause_time)
    
    return result_df

In [ ]:
result_df1 = add_parameter_columns(df1, eq_data, checkpoint_file=CHECKPOINT_FILE)
result_df1

In [ ]:
result_df2 = add_parameter_columns(df2, eq_data, checkpoint_file=RAG_CHECKPOINT_FILE)
result_df2

### 🌍 Prompt Generation

This module generates corresponding prompts for downstream LLM-based reasoning.


#### ⚙️ Functions

##### `generate_earthquake_prompts()`

Generates structured LLM prompts from a dataset enriched with earthquake and sociodemographic parameters.

- Initializes two prompt columns:
  - `system_prompt`: Static context prompt for LLMs
  - `earthquake_prompt`: Dynamic prompt generated from row-level variables
- Resumes prompt generation from checkpoint

In [ ]:
def generate_earthquake_prompts(df, checkpoint_file=CHECKPOINT_FILE):
    """
    Generates system prompts and earthquake prompts using the parameter columns already added to the dataframe.
    """
    # Load checkpoint
    start_idx = 0
    result_df = df.copy()
    
    # Initialize prompt columns
    if 'system_prompt' not in result_df.columns:
        result_df['system_prompt'] = None
    
    if 'earthquake_prompt' not in result_df.columns:
        result_df['earthquake_prompt'] = None
    
    # Add system prompt
    if result_df['system_prompt'].isnull().all():
        result_df['system_prompt'] = SYSTEM_PROMPT
    
    if os.path.exists(checkpoint_file):
        try:
            with open(checkpoint_file, 'rb') as f:
                checkpoint = pickle.load(f)
                result_df = checkpoint['df']
                start_idx = checkpoint['last_processed_idx'] + 1
                print(f"Resuming prompt generation from index {start_idx}")
        except Exception as e:
            print(f"Error loading prompt checkpoint: {e}")
    
    total_rows = len(df)
    for index in range(start_idx, total_rows):
        row = result_df.iloc[index]
        
        print(f"\nGenerating prompt {index}/{total_rows}")
        
        # Generate earthquake prompt
        try:
            prompt_params = {
                "eq_place": row['eq_place'],
                "eq_lat": round(row['eq_lat'], 3) if row['eq_lat'] is not None else "information is not available.",
                "eq_lng": round(row['eq_lng'], 3) if row['eq_lng'] is not None else "information is not available.",
                "eq_magnitude": round(row['eq_magnitude'], 1) if row['eq_magnitude'] is not None else "information is not available.",
                
                "state": row['State/Region'], 
                "city": row['City'],
                "zipcode": row['Zip Code'],
                "lat": round(row['latitude'], 3),
                "lng": round(row['longitude'], 3),
                "distance": round(row['distance'], 2) if row['distance'] is not None else "information is not available.",
                "vs30": row['vs30'] if row['vs30'] is not None else "information is not available.",
                
                "population_density": round(row['population_density'], 2) if row['population_density'] is not None else "information is not available.",
                "urban_population_pct": round(row['urban_population_pct'], 2) if row['urban_population_pct'] is not None else "information is not available.",
                "median_household_income": int(round(row['median_household_income'], 0)) if row['median_household_income'] is not None else "information is not available.",
                "education": round(row['education'], 2) if row['education'] is not None else "information is not available.",
                "over_65_rate": round(row['over_65_rate'], 2) if row['over_65_rate'] is not None else "information is not available.",

                "building": row['building'] if row['building'] is not None else "Building information is not available."
            }
            
            result_df.loc[index, 'earthquake_prompt'] = EARTHQUAKE_PROMPT.format(**prompt_params)
        except Exception as e:
            print(f"Error generating prompt for index {index}: {str(e)}")
        
        # Save checkpoint after each row
        checkpoint = {
            'df': result_df,
            'last_processed_idx': index
        }
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint, f)
    
    return result_df

In [ ]:
result_df1 = generate_earthquake_prompts(result_df1, checkpoint_file=CHECKPOINT_FILE)
result_df1.to_csv(OUTPUT_SAMPLES_CSV,index=False)

In [ ]:
result_df2 = generate_earthquake_prompts(result_df2, checkpoint_file=RAG_CHECKPOINT_FILE)
result_df2.to_csv(OUTPUT_RAG_SAMPLES_CSV,index=False)